<a href="https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face secret created successfully.")

Paste your Hugging Face READ token: ··········
Hugging Face secret created successfully.


In [2]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT *
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


## Unit of Analysis + Time Window

One row represents one client in the `dim_clients` table.

For this assignment, I will work with a mid-panel time window (March 2026) when querying the warehouse, as recommended in the instructions. This avoids using the final month as a development dataset.

The unit of analysis is one client record with its Search Console and Google Analytics access information.

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT *
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


## Fields

### Features
- has_gsc_access
- has_ga4_access
- access_profile
- gsc_data_start
- ga4_data_start

### Label / Proxy
- Client access readiness (proxy)

### Context
- client_created_date
- client_updated_date

### Excluded
- client_hash_id

Reason: The client hash is only an identifier and does not provide useful predictive information.

In [4]:
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 10
""").df()

print(df.columns.tolist())

['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## Verify it with Queries

The following queries verify the data contract:

1. Verify the grain (one row represents one client).
2. Count the total number of client records.
3. Check data availability and missing values for Search Console and Google Analytics access.

In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"

# Query 1: Verify grain
print("Query 1: Verify Grain")
display(con.sql(f"""
SELECT *
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 5
""").df())

# Query 2: Count total clients
print("\nQuery 2: Total Clients")
display(con.sql(f"""
SELECT COUNT(*) AS total_clients
FROM read_parquet('{rel}/dim_clients.parquet')
""").df())

# Query 3: Availability (IS TRUE)
print("\nQuery 3: GSC & GA4 Availability")
display(con.sql(f"""
SELECT
COUNT(*) AS total_clients,
COUNT(*) FILTER (WHERE has_gsc_access IS TRUE) AS gsc_available,
COUNT(*) FILTER (WHERE has_ga4_access IS TRUE) AS ga4_available
FROM read_parquet('{rel}/dim_clients.parquet')
""").df())

Query 1: Verify Grain


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT



Query 2: Total Clients


,total_clients
0,104



Query 3: GSC & GA4 Availability


,total_clients,gsc_available,ga4_available
0,104,67,54


## Data Limits

This dataset has several limitations.

- Client histories are unbalanced because different clients joined at different times.
- Some early records contain only Google Search Console data and do not include Google Analytics data.
- The dataset is observational and supports decision-making, but it cannot prove causal relationships.
- Window overlaps and incomplete historical coverage may affect long-term comparisons.

In [6]:
print("Data limitations reviewed and documented.")

Data limitations reviewed and documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.